# Week 55, Day 6: Forge: Week 55 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w055/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(1 / (1 - 0.9 ** 2), 2))
print("2.", 10 / (10 + 30))
print("3.", np.sign([2.0, -0.3, 0.7]) * np.maximum(np.abs([2.0, -0.3, 0.7]) - 0.5, 0))

### Exercise: Part A: least squares, ridge, lasso, elastic net

The fitting functions and the cross-validation helper are provided. For each penalised estimator, choose lambda by cv_lambda over its grid, refit on all 150 training trials, and store (test R², number of non-zero weights) in results.

In [ ]:
import numpy as np
r = np.random.default_rng(556)
G, S = 12, 5
P = G * S

def population(n):
    shared = r.normal(size=(n, G))
    return np.repeat(shared, S, axis=1) + 0.5 * r.normal(size=(n, P))

w_true = np.zeros(P); w_true[:S] = 0.3; w_true[S:2 * S] = -0.2
X, X_te = population(150), population(5000)
y, y_te = X @ w_true + r.normal(size=150), X_te @ w_true + r.normal(size=5000)
mu, sd = X.mean(0), X.std(0); Xs, Xs_te = (X - mu) / sd, (X_te - mu) / sd
ym = y.mean()

def soft(z, l):
    return np.sign(z) * np.maximum(np.abs(z) - l, 0)

def enet(X, y, lam, alpha, iters=40):
    n = len(y); w = np.zeros(X.shape[1]); res = y.copy()
    for _ in range(iters):
        for j in range(X.shape[1]):
            rho = X[:, j] @ (res + X[:, j] * w[j]) / n
            new = soft(rho, lam * alpha) / (1 + lam * (1 - alpha))
            res -= X[:, j] * (new - w[j]); w[j] = new
    return w

def ridge(X, y, lam):
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)

def cv_lambda(fitter, grid, K=5):
    folds = np.array_split(np.arange(150), K); errs = []
    for lam in grid:
        e = 0.0
        for te in folds:
            tr = np.setdiff1d(np.arange(150), te)
            e += np.sum((y[te] - ym - Xs[te] @ fitter(Xs[tr], y[tr] - ym, lam)) ** 2)
        errs.append(e)
    return grid[int(np.argmin(errs))]

r2 = lambda w: 1 - np.mean((y_te - ym - Xs_te @ w) ** 2) / y_te.var()
fitters = {"ridge": (ridge, np.logspace(-1, 4, 12)),
           "lasso": (lambda A, b, l: enet(A, b, l, 1.0), np.logspace(-3, -0.5, 10)),
           "elastic net": (lambda A, b, l: enet(A, b, l, 0.3), np.logspace(-3, 0, 10))}
w_ols = np.linalg.lstsq(Xs, y - ym, rcond=None)[0]
results = {"least squares": (r2(w_ols), int(np.sum(w_ols != 0)))}
for name, (fitter, grid) in fitters.items():
    lam = cv_lambda(fitter, grid)
    w = ...
    results[name] = ...
for name, (score, nz) in results.items():
    print(f"{name:14s} test R² {score:.3f}, neurons used {nz} (10 truly matter)")

In [ ]:
import numpy as np
def test_all():
    "results has test R² and number of non-zero weights for the four estimators"
    assert set(results) == {"least squares", "ridge", "lasso", "elastic net"}
def test_order():
    "every penalised estimator beats least squares on test R²"
    assert all(results[k][0] > results["least squares"][0] for k in ["ridge", "lasso", "elastic net"])
def test_sparse():
    "the lasso uses no more neurons than the elastic net, which uses fewer than ridge"
    assert results["lasso"][1] <= results["elastic net"][1] < results["ridge"][1]

test_all()
test_order()
test_sparse()
print("All checks passed")

### Exercise: Part B: selection frequencies under the bootstrap

For 30 bootstrap resamples (trials drawn with replacement), fit the lasso at lambda = 0.15 and record which weights are non-zero. Store the fraction of resamples in which each neuron is selected in freq.

In [ ]:
B, lam_b = 30, 0.15
counts = np.zeros(P)
for b in range(B):
    idx = r.integers(0, 150, 150)
    w = enet(Xs[idx], y[idx] - y[idx].mean(), lam_b, 1.0, iters=30)
    counts += ...
freq = counts / B
print("selection frequency, group 1 (true):", np.round(freq[:S], 2))
print("selection frequency, group 2 (true):", np.round(freq[S:2 * S], 2))
print(f"mean frequency of the 50 null neurons: {freq[2 * S:].mean():.2f}; selected in over 80% of resamples: {np.flatnonzero(freq > 0.8)}")

In [ ]:
import numpy as np
def test_freq():
    "freq has one selection frequency per neuron, between 0 and 1"
    assert freq.shape == (P,) and np.all((freq >= 0) & (freq <= 1))
def test_groups():
    "neurons in the two true groups are selected more often, on average, than the rest"
    assert freq[:2 * S].mean() > 2 * freq[2 * S:].mean()

test_freq()
test_groups()
print("All checks passed")